In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [4]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+s://fa687069.databases.neo4j.io",
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="fa687069",
    refresh_schema=False,
)
graph

In [5]:
movie_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv' AS row

MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

WITH m, row
UNWIND split(row.director, '|') AS director
MERGE (d:Person {name: trim(director)})
MERGE (d)-[:DIRECTED]->(m)

WITH m, row
UNWIND split(row.actors, '|') AS actor
MERGE (a:Person {name: trim(actor)})
MERGE (a)-[:ACTED_IN]->(m)

WITH m, row
UNWIND split(row.genres, '|') AS genre
MERGE (g:Genre {name: trim(genre)})
MERGE (m)-[:IN_GENRE]->(g)
"""

In [6]:
graph._driver.session(database=graph._database).run(movie_query)

In [7]:
graph.refresh_schema()

In [8]:
graph.get_schema

'Node properties:\nMovie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}\nPerson {name: STRING}\nGenre {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:Person)-[:DIRECTED]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)'

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=GROQ_API_KEY)

In [10]:
from langchain.chains import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True)

chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001AF8A9264A0>, cypher_generation_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'schema'], template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001AFBDB047C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AFBDB046D0>, model_name='openai/gpt-oss-120b', groq_api_key=SecretStr('******

In [11]:
response = chain.invoke({"query": "Who was the director of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Jumanji"}) RETURN p.name
Full Context:
[{'p.name': 'Joe Johnston'}]

> Finished chain.


{'query': 'Who was the director of the movie Jumanji',
 'result': 'Joe Johnston was the director of the movie Jumanji.'}

In [12]:
response = chain.invoke({"query": "Who was the actors of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Jumanji'})
RETURN p.name;
Full Context:
[{'p.name': 'Robin Williams'}, {'p.name': 'Bradley Pierce'}, {'p.name': 'Kirsten Dunst'}, {'p.name': 'Jonathan Hyde'}]

> Finished chain.


{'query': 'Who was the actors of the movie Jumanji',
 'result': 'Robin\u202fWilliams, Bradley\u202fPierce, Kirsten\u202fDunst, Jonathan\u202fHyde.'}

In [13]:
response = chain.invoke({"query": "What is the genre of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Jumanji'})-[:IN_GENRE]->(g:Genre)
RETURN g.name AS genre
Full Context:
[{'genre': 'Adventure'}, {'genre': 'Children'}, {'genre': 'Fantasy'}]

> Finished chain.


{'query': 'What is the genre of the movie Jumanji',
 'result': 'Adventure, Children, Fantasy'}

### Prompt Strategies


In [14]:
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie {{title: 'Schindler\\'s List'}})-[:IN_GENRE]->(g:Genre) RETURN g.name",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three different actors named 'John'?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Identify movies where directors also played a role in the film.",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor with the highest number of movies in the database.",
        "query": "MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
]

In [15]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

prompt_template = PromptTemplate.from_template(
    "User input: {question} \n Cypher query: {query}"
)

prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=prompt_template,
    prefix="You are a Neo4J expert. Given a input question, create a syntactically very accurate Cypher query.",
    suffix="User input: {question} \n Cypher query: ",
    input_variables=["question", "schema"],
)

prompt

FewShotPromptTemplate(input_variables=['question'], examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)"}, {'question': "List all the genres of the movie Schindler's List", 'query': "MATCH (m:Movie {{title: 'Schindler\\'s List'}})-[:IN_GENRE]->(g:Genre) RETURN g.name"}, {'question': 'Which actors have worked in movies from both the comedy and action genres?', 'query': "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name"}], example_prompt=PromptTemplate(input_variables=['query', 'question'], template='

In [16]:
print(prompt.format(question="How many artists are there?", schema="Foo"))

You are a Neo4J expert. Given a input question, create a syntactically very accurate Cypher query.

User input: How many artists are there? 
 Cypher query: MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User input: Which actors played in the movie Casino? 
 Cypher query: MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name

User input: How many movies has Tom Hanks acted in? 
 Cypher query: MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)

User input: List all the genres of the movie Schindler's List 
 Cypher query: MATCH (m:Movie {title: 'Schindler\'s List'})-[:IN_GENRE]->(g:Genre) RETURN g.name

User input: Which actors have worked in movies from both the comedy and action genres? 
 Cypher query: MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name

User input: How many artists are there? 
 Cypher 

In [17]:
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, cypher_prompt=prompt, verbose=True)

In [18]:
chain.invoke("List all the genres of the movie Toy Story")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Toy Story'})-[:IN_GENRE]->(g:Genre)
RETURN g.name
Full Context:
[{'g.name': 'Adventure'}, {'g.name': 'Animation'}, {'g.name': 'Children'}, {'g.name': 'Comedy'}, {'g.name': 'Fantasy'}]

> Finished chain.


{'query': 'List all the genres of the movie Toy Story',
 'result': 'Adventure, Animation, Children, Comedy, Fantasy.'}

In [19]:
chain.invoke("How many movies Tom Hanks acted in ?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN count(m) AS movieCount
Full Context:
[{'movieCount': 2}]

> Finished chain.


{'query': 'How many movies Tom Hanks acted in ?',
 'result': 'Tom Hanks acted in 2 movies.'}

In [20]:
chain.invoke("Display the movies Tom Hanks acted in")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN m.title

Full Context:
[{'m.title': 'Toy Story'}, {'m.title': 'Apollo 13'}]

> Finished chain.


{'query': 'Display the movies Tom Hanks acted in',
 'result': 'Toy Story, Apollo 13'}

In [24]:
chain.invoke("Which directors have made movies with at least three different actors?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
// Directors who have directed at least one movie that features three or more distinct actors
MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person)
WITH d, m, count(DISTINCT a) AS actorCount
WHERE actorCount >= 3
RETURN DISTINCT d.name AS director

Full Context:
[{'director': 'John Lasseter'}, {'director': 'Joe Johnston'}, {'director': 'Howard Deutch'}, {'director': 'Forest Whitaker'}, {'director': 'Charles Shyer'}, {'director': 'Michael Mann'}, {'director': 'Sydney Pollack'}, {'director': 'Peter Hewitt'}, {'director': 'Peter Hyams'}, {'director': 'Martin Campbell'}]

> Finished chain.


{'query': 'Which directors have made movies with at least three different actors?',
 'result': 'I don’t know the answer.'}